# M6 VitalDB ART/NIBP feasibility audit
This notebook uses the public VitalDB surgical dataset. It checks patient-disjoint candidate cases and real waveform/cuff-value coverage; it does **not** train or score M6. Track presence is not signal quality, a displayed cuff value is not a verified cuff measurement time, and surgical cases are not a ward validation set. Run the three code cells in order. The last cell can take several minutes and prints a running ETA. No raw patient signals are saved to Drive or GitHub.


In [ ]:
import sys, subprocess, pathlib
from google.colab import drive
drive.mount('/content/drive')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vitaldb>=1.7,<2'], check=True)
repo = pathlib.Path('/content/sentinelward-ml-m6')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ADI238IND/sentinelward-ml.git', str(repo)], check=True)
print('Repository ready:', repo)


In [ ]:
import json, subprocess, sys, pathlib
out = pathlib.Path('/content/drive/MyDrive/SentinelWard-work/m6/vitaldb_metadata_audit.json')
subprocess.run([sys.executable, '-m', 'examples.m6_vitaldb_audit', '--waveform-cases', '0', '--output', str(out)], cwd=repo, check=True)
r = json.loads(out.read_text())
print('Candidate paired-track cases:', r['paired_track_case_count'])
print('Patient-disjoint split:', r['split_case_counts'])
print('Twelve candidate training cases:', r['candidate_12_bed_caseids'])
print('No training or live score has occurred.')


In [ ]:
import json, subprocess, sys, pathlib
out = pathlib.Path('/content/drive/MyDrive/SentinelWard-work/m6/vitaldb_waveform_audit.json')
print('Auditing 12 actual cases, one at a time; ETA will update after each case.')
subprocess.run([sys.executable, '-m', 'examples.m6_vitaldb_audit', '--waveform-cases', '12', '--output', str(out)], cwd=repo, check=True)
r = json.loads(out.read_text())
print('Completed cases:', len(r['waveform_audit']))
print('Cases with at least one technical ART+displayed-NIBP window:', sum(x['windows_with_art_and_a_displayed_nibp_value'] > 0 for x in r['waveform_audit']))
print('M6 live score authorized:', r['m6_live_score_authorized'])
print('Audit saved:', out)
